**Objetivo:** Carregar os dados brutos do SIH e CNES, filtrar internações por IAM (CID I21), realizar limpeza e integrar as duas bases.

**Outputs gerados:**
- `data/interim/sih_iam.parquet` — internações IAM limpas
- `data/interim/cnes_hospitais.parquet` — dados hospitalares consolidados
- `data/processed/base_modelagem.parquet` — base final (SIH + CNES)

## Configuração do ambiente

In [67]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
import sys
import json

In [68]:
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: /home/carolina/Documents/TCC Documentos/TCC


In [69]:
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

#  Caminhos 
RAW_SIH     = Path(ROOT,'data/input/SIH')
RAW_CNES    = Path(ROOT,'data/input/CNES')
INTERIM     = Path(ROOT,'data/interim')
PROCESSED   = Path(ROOT,'data/processed')

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# CIDs de interesse: I21 e seus subcódigos (I21.0 a I21.9)
CID_IAM = ['I21', 'I210', 'I211', 'I212', 'I213', 'I214',
            'I219', 'I21.0', 'I21.1', 'I21.2', 'I21.3',
            'I21.4', 'I21.9']

# Separador dos CSVs do pysus (ajuste se necessário)
SEP = ';'

# Encoding padrão DATASUS
ENCODING = 'latin-1'

print('✓ Configurações carregadas')
print(f'  SIH  → {RAW_SIH}')
print(f'  CNES → {RAW_CNES}')

✓ Configurações carregadas
  SIH  → /home/carolina/Documents/TCC Documentos/TCC/data/input/SIH
  CNES → /home/carolina/Documents/TCC Documentos/TCC/data/input/CNES


In [70]:
arquivos_sih = sorted(glob.glob(str(RAW_SIH / '*.csv')))
print(f'Arquivos SIH encontrados: {len(arquivos_sih)}')
for f in arquivos_sih:
    print(f'  {Path(f).name}')

Arquivos SIH encontrados: 12
  rdsp2501.csv
  rdsp2502.csv
  rdsp2503.csv
  rdsp2504.csv
  rdsp2505.csv
  rdsp2506.csv
  rdsp2507.csv
  rdsp2508.csv
  rdsp2509.csv
  rdsp2510.csv
  rdsp2511.csv
  rdsp2512.csv


### removendo colunas e atualizando nomes

In [71]:
path_dicionario =  Path(ROOT,'data/external/dicionario_SIH.json')
path_dicionario
with open(path_dicionario, "r", encoding="utf-8") as f:
    schema = json.load(f)

In [72]:
rename_dict = {
    col["old_name"]: col["new_name"]
    for col in schema
}
rename_dict


{'UF_ZI': 'municipio_gestor',
 'ANO_CMPT': 'ano_competencia',
 'MES_CM': 'mes_competencia',
 'ESPEC': 'especialidade_leito',
 'CGC_HOSP': 'cnpj_hospital',
 'N_AIH': 'numero_aih',
 'IDENT': 'tipo_aih',
 'CEP': 'cep_paciente',
 'MUNIC_RES': 'municipio_residencia',
 'NASC': 'data_nascimento',
 'SEXO': 'sexo',
 'UTI_MES_IN': 'uti_mes_inicial',
 'UTI_MES_AN': 'uti_mes_anterior',
 'UTI_MES_AL': 'uti_mes_alta',
 'UTI_MES_TO': 'uti_mes_total',
 'MARCA_UTI': 'tipo_uti',
 'UTI_INT_IN': 'uti_intermediaria_inicial',
 'UTI_INT_AN': 'uti_intermediaria_anterior',
 'UTI_INT_AL': 'uti_intermediaria_alta',
 'UTI_INT_TO': 'uti_intermediaria_total',
 'DIAR_ACOM': 'diarias_acompanhante',
 'QT_DIARIAS': 'quantidade_diarias',
 'PROC_SOLIC': 'procedimento_solicitado',
 'PROC_REA': 'procedimento_realizado',
 'VAL_SH': 'valor_servicos_hospitalares',
 'VAL_SP': 'valor_servicos_profissionais',
 'VAL_SADT': 'valor_sadt',
 'VAL_RN': 'valor_rn',
 'VAL_ACOMP': 'valor_acompanhante',
 'VAL_ORTP': 'valor_ortese_protese'

In [73]:
df = pd.read_csv(arquivos_sih[0])
df.head(5)

/tmp/ipykernel_1408367/906988290.py:1: DtypeWarning: Columns (84,87,88,96,97,98,99,100,101,102,103) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arquivos_sih[0])


,UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,SEXO,UTI_MES_IN,UTI_MES_AN,UTI_MES_AL,UTI_MES_TO,MARCA_UTI,UTI_INT_IN,UTI_INT_AN,UTI_INT_AL,UTI_INT_TO,DIAR_ACOM,QT_DIARIAS,PROC_SOLIC,PROC_REA,VAL_SH,VAL_SP,VAL_SADT,VAL_RN,VAL_ACOMP,VAL_ORTP,...,RACA_COR,ETNIA,SEQUENCIA,REMESSA,AUD_JUST,SIS_JUST,VAL_SH_FED,VAL_SP_FED,VAL_SH_GES,VAL_SP_GES,VAL_UCI,MARCA_UCI,DIAGSEC1,DIAGSEC2,DIAGSEC3,DIAGSEC4,DIAGSEC5,DIAGSEC6,DIAGSEC7,DIAGSEC8,DIAGSEC9,TPDISEC1,TPDISEC2,TPDISEC3,TPDISEC4,TPDISEC5,TPDISEC6,TPDISEC7,TPDISEC8,TPDISEC9
0,350000,2025,1,1,46374500028366.00,3525100117847,1,11704840,354100,19840716,3,0,0,0,0,0,0,0,0,0,0,1,407040129,407040129,298.55,136.44,0.00,0.00,0.00,0.00,...,1,0,28722,HE35000001N202501.DTS,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0,Z540,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0
1,350000,2025,1,2,46374500028366.00,3524130275908,1,11741802,352210,20070606,3,0,0,0,0,0,0,0,0,0,3,3,310010039,310010039,361.25,275.57,0.00,0.00,0.00,0.00,...,3,0,28828,HE35000001N202501.DTS,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0,O700,Z356,O721,O365,Z353,NaN,NaN,NaN,NaN,1,1,1,1,1,0,0,0,0
2,350000,2025,1,2,46374500028366.00,3524130278427,1,11730000,353110,20030120,3,0,0,0,0,0,0,0,0,0,0,4,310010039,310010039,430.53,275.57,0.00,0.00,0.00,0.00,...,1,0,28829,HE35000001N202501.DTS,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0,E039,O702,O48,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,0,0,0,0,0,0
3,350000,2025,1,2,46374500028366.00,3524130278449,1,11743250,352210,20030405,3,0,0,0,0,0,0,0,0,0,0,5,310010039,310010039,353.44,279.44,0.00,0.00,0.00,0.00,...,1,0,28830,HE35000001N202501.DTS,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0,O16,E039,O700,O234,NaN,NaN,NaN,NaN,NaN,1,1,1,1,0,0,0,0,0
4,350000,2025,1,2,46374500028366.00,3524130278526,1,11730000,353110,20050224,3,0,0,0,0,0,0,0,0,0,0,4,310010039,310010039,337.25,275.57,0.00,0.00,0.00,0.00,...,1,0,28831,HE35000001N202501.DTS,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0,E039,N200,O701,O691,NaN,NaN,NaN,NaN,NaN,1,1,1,1,0,0,0,0,0


In [74]:
df = df.rename(columns=rename_dict)
df.head(5)

,municipio_gestor,ano_competencia,MES_CMPT,especialidade_leito,cnpj_hospital,numero_aih,tipo_aih,cep_paciente,municipio_residencia,data_nascimento,sexo,uti_mes_inicial,uti_mes_anterior,uti_mes_alta,uti_mes_total,tipo_uti,uti_intermediaria_inicial,uti_intermediaria_anterior,uti_intermediaria_alta,uti_intermediaria_total,diarias_acompanhante,quantidade_diarias,procedimento_solicitado,procedimento_realizado,valor_servicos_hospitalares,valor_servicos_profissionais,valor_sadt,valor_rn,valor_acompanhante,valor_ortese_protese,...,raca_cor,etnia,sequencial_remessa,numero_remessa,justificativa_auditor,justificativa_estabelecimento,valor_sh_federal,valor_sp_federal,valor_sh_gestor,valor_sp_gestor,valor_uci,tipo_uci,diagnostico_secundario_1,diagnostico_secundario_2,diagnostico_secundario_3,diagnostico_secundario_4,diagnostico_secundario_5,diagnostico_secundario_6,diagnostico_secundario_7,diagnostico_secundario_8,diagnostico_secundario_9,tipo_diag_sec_1,tipo_diag_sec_2,tipo_diag_sec_3,tipo_diag_sec_4,tipo_diag_sec_5,tipo_diag_sec_6,tipo_diag_sec_7,tipo_diag_sec_8,tipo_diag_sec_9
0,350000,2025,1,1,46374500028366.00,3525100117847,1,11704840,354100,19840716,3,0,0,0,0,0,0,0,0,0,0,1,407040129,407040129,298.55,136.44,0.00,0.00,0.00,0.00,...,1,0,28722,HE35000001N202501.DTS,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0,Z540,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0
1,350000,2025,1,2,46374500028366.00,3524130275908,1,11741802,352210,20070606,3,0,0,0,0,0,0,0,0,0,3,3,310010039,310010039,361.25,275.57,0.00,0.00,0.00,0.00,...,3,0,28828,HE35000001N202501.DTS,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0,O700,Z356,O721,O365,Z353,NaN,NaN,NaN,NaN,1,1,1,1,1,0,0,0,0
2,350000,2025,1,2,46374500028366.00,3524130278427,1,11730000,353110,20030120,3,0,0,0,0,0,0,0,0,0,0,4,310010039,310010039,430.53,275.57,0.00,0.00,0.00,0.00,...,1,0,28829,HE35000001N202501.DTS,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0,E039,O702,O48,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,0,0,0,0,0,0
3,350000,2025,1,2,46374500028366.00,3524130278449,1,11743250,352210,20030405,3,0,0,0,0,0,0,0,0,0,0,5,310010039,310010039,353.44,279.44,0.00,0.00,0.00,0.00,...,1,0,28830,HE35000001N202501.DTS,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0,O16,E039,O700,O234,NaN,NaN,NaN,NaN,NaN,1,1,1,1,0,0,0,0,0
4,350000,2025,1,2,46374500028366.00,3524130278526,1,11730000,353110,20050224,3,0,0,0,0,0,0,0,0,0,0,4,310010039,310010039,337.25,275.57,0.00,0.00,0.00,0.00,...,1,0,28831,HE35000001N202501.DTS,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0,E039,N200,O701,O691,NaN,NaN,NaN,NaN,NaN,1,1,1,1,0,0,0,0,0


In [75]:
cols_keep = [
    "especialidade_leito",
    "numero_aih",
    "data_nascimento",
    "sexo",
    "uti_mes_total",
    "tipo_uti",
    "procedimento_solicitado",
    "procedimento_realizado",
    "data_internacao",
    "data_saida",
"diagnostico_principal",
"diagnostico_secundario",
"motivo_saida",
"codigo_idade",
"idade",
"dias_permanencia",
"indicador_obito",
"carater_internacao",
"cid_notificacao",
"cnes",
"cid_associado",
"cid_morte",
"complexidade",
"raca_cor",
"etnia",
]

df_col = df[cols_keep]
df_col

,especialidade_leito,numero_aih,data_nascimento,sexo,uti_mes_total,tipo_uti,procedimento_solicitado,procedimento_realizado,data_internacao,data_saida,diagnostico_principal,diagnostico_secundario,motivo_saida,codigo_idade,idade,dias_permanencia,indicador_obito,carater_internacao,cid_notificacao,cnes,cid_associado,cid_morte,complexidade,raca_cor,etnia
0,1,3525100117847,19840716,3,0,0,407040129,407040129,20250116,20250117,K429,0,12,4,40,1,0,1,NaN,2087804,0,0,2,1,0
1,2,3524130275908,20070606,3,0,0,310010039,310010039,20241205,20241208,O800,0,62,4,17,3,0,2,NaN,2087804,0,0,2,3,0
2,2,3524130278427,20030120,3,0,0,310010039,310010039,20241212,20241216,O800,0,62,4,21,4,0,2,NaN,2087804,0,0,2,1,0
3,2,3524130278449,20030405,3,0,0,310010039,310010039,20241212,20241217,O800,0,61,4,21,5,0,2,NaN,2087804,0,0,2,1,0
4,2,3524130278526,20050224,3,0,0,310010039,310010039,20241212,20241216,O800,0,62,4,19,4,0,2,NaN,2087804,0,0,2,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238139,1,3525109544231,19700728,3,0,0,409070050,409070050,20250121,20250123,N811,0,12,4,54,2,0,1,NaN,2087618,0,0,2,1,0
238140,2,3525109544924,19960528,3,0,0,310010039,310010039,20250113,20250114,O809,0,61,4,28,1,0,2,NaN,2087618,0,0,2,3,0
238141,2,3525109544957,20100415,3,0,0,310010039,310010039,20250116,20250118,O809,0,61,4,14,2,0,2,NaN,2087618,0,0,2,3,0
238142,2,3525109544979,19880921,3,0,0,310010039,310010039,20250111,20250113,O809,0,61,4,36,2,0,2,NaN,2087618,0,0,2,3,0


### filtrando pelo CID

In [87]:
df_col["diagnostico_principal"] = df_col["diagnostico_principal"].astype(str)
df_col["diagnostico_secundario"] = df_col["diagnostico_secundario"].astype(str)


df_iam = df_col[
    df_col["diagnostico_principal"].str.startswith("I21") |
    df_col["diagnostico_secundario"].str.startswith("I21")
]
df_iam.head(20)

/tmp/ipykernel_1408367/1967526092.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_col["diagnostico_principal"] = df_col["diagnostico_principal"].astype(str)
/tmp/ipykernel_1408367/1967526092.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_col["diagnostico_secundario"] = df_col["diagnostico_secundario"].astype(str)


,especialidade_leito,numero_aih,data_nascimento,sexo,uti_mes_total,tipo_uti,procedimento_solicitado,procedimento_realizado,data_internacao,data_saida,diagnostico_principal,diagnostico_secundario,motivo_saida,codigo_idade,idade,dias_permanencia,indicador_obito,carater_internacao,cid_notificacao,cnes,cid_associado,cid_morte,complexidade,raca_cor,etnia
183,1,3524128896860,19740525,1,3,86,406030049,406030049,20241217,20241219,I210,0,27,4,50,2,0,2,NaN,2077396,0,0,3,3,0
184,1,3524128896871,19631015,3,0,0,406030030,406030030,20241211,20241212,I219,0,12,4,61,1,0,2,NaN,2077396,0,0,3,3,0
195,1,3524131709505,19550715,3,2,86,406030022,406030022,20241226,20241228,I219,0,41,4,69,2,1,2,NaN,2077396,0,0,3,1,0
205,1,3524128891019,19510501,3,2,86,406030049,406030049,20241212,20241215,I213,0,51,4,73,3,0,2,NaN,2077396,0,0,3,2,0
207,1,3524128891437,19600521,1,5,86,406030049,406030049,20241213,20241217,I219,0,27,4,64,4,0,2,NaN,2077396,0,0,3,1,0
220,1,3524131703587,19470808,1,0,0,406030022,406030022,20241221,20241227,I219,0,12,4,77,6,0,2,NaN,2077396,0,0,3,1,0
252,1,3524131713850,19701025,3,0,0,406030030,406030030,20241230,20250101,I213,0,27,4,54,2,0,2,NaN,2077396,0,0,3,3,0
262,1,3524128892394,19551101,1,0,0,406030049,406030049,20241214,20241218,I213,0,12,4,69,4,0,2,NaN,2077396,0,0,3,1,0
271,1,3524131704082,19720706,1,5,86,406030049,406030049,20241221,20241226,I210,0,12,4,52,5,0,2,NaN,2077396,0,0,3,2,0
281,1,3524131711199,19610902,3,3,86,406010935,406010935,20241226,20241231,I219,0,12,4,63,5,0,2,NaN,2077396,0,0,3,1,0


### verificação de nulos

In [85]:
df_iam = df_iam.replace(["", "0000", "0", "000"], pd.NA)

In [86]:
nulls = pd.DataFrame({
    "qtd_nulos": df_iam.isnull().sum(),
    "perc_nulos": df_iam.isnull().mean() * 100
}).sort_values(by="perc_nulos", ascending=False)

nulls

,qtd_nulos,perc_nulos
diagnostico_secundario,4110,100.00
cid_notificacao,4110,100.00
etnia,65,1.58
numero_aih,0,0.00
uti_mes_total,0,0.00
tipo_uti,0,0.00
data_nascimento,0,0.00
sexo,0,0.00
especialidade_leito,0,0.00
data_internacao,0,0.00
